# Phase 3: Cluster Descriptions via Pegasus

Generate human-readable labels for each cluster using Twelve Labs Pegasus 1.2.

**Prerequisites:** Run `01_embeddings.py` and `02_clustering.py` first so the dataset has `embedding`, `cluster_id`, and `centroid_distance` fields.

**Strategy:**
1. For each cluster, find the 2 samples closest to the centroid
2. Upload each representative video as a Twelve Labs asset
3. Call Pegasus to generate a one-sentence description
4. Combine descriptions into a cluster label
5. Store `cluster_label` on every sample in the dataset

## 1. Setup & Imports

In [1]:
import os
import time
from datetime import datetime
from collections import defaultdict

import fiftyone as fo
from twelvelabs import TwelveLabs
from twelvelabs.types import VideoContext_AssetId
from twelvelabs.indexes import IndexesCreateRequestModelsItem

# Validate API key
api_key = os.environ.get("TWELVELABS_API_KEY")
assert api_key, "Set TWELVELABS_API_KEY environment variable first"
client = TwelveLabs(api_key=api_key)
print("Twelve Labs client ready")

/Users/rishimule/Desktop/GitHub/video-content-gap-analyzer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Twelve Labs client ready


## 2. Load Dataset & Validate Fields

The dataset should already have `cluster_id` and `centroid_distance` from Phase 2.

In [2]:
dataset = fo.load_dataset("Voxel51/Safe_and_Unsafe_Behaviours")
print(f"Dataset: {dataset.name} ({len(dataset)} samples)")

# Verify clustering fields
sample = dataset.first()
assert sample["cluster_id"] is not None, "Run 02_clustering.py first"
assert sample["centroid_distance"] is not None, "Run 02_clustering.py first"
print("Clustering fields verified")

Dataset: Voxel51/Safe_and_Unsafe_Behaviours (200 samples)
Clustering fields verified


## 3. Find Cluster Representatives

For each cluster, select the 2 samples with the smallest centroid distance — these are most representative of the cluster's content.

In [3]:
REPS_PER_CLUSTER = 2

clusters = defaultdict(list)
for sample in dataset:
    try:
        cid = sample["cluster_id"]
        dist = sample["centroid_distance"]
    except (KeyError, AttributeError):
        continue
    if cid is not None and dist is not None:
        clusters[cid].append((dist, sample))

representatives = {}
for cid in sorted(clusters.keys()):
    sorted_samples = sorted(clusters[cid], key=lambda x: x[0])
    representatives[cid] = [s for _, s in sorted_samples[:REPS_PER_CLUSTER]]

print(f"Found {len(representatives)} clusters\n")
for cid, reps in representatives.items():
    dists = [f"{r['centroid_distance']:.6f}" for r in reps]
    filenames = [os.path.basename(r.filepath) for r in reps]
    print(f"  Cluster {cid}: {len(reps)} reps")
    for fn, d in zip(filenames, dists):
        print(f"    {fn} (dist={d})")

Found 2 clusters

  Cluster 0: 2 reps
    0_tr103.mp4 (dist=0.001377)
    0_tr100.mp4 (dist=0.002224)
  Cluster 1: 2 reps
    1_tr13.mp4 (dist=0.002269)
    1_tr14.mp4 (dist=0.003039)


## 4. Generate Pegasus Descriptions

Upload each representative video as a Twelve Labs asset, then call Pegasus to describe it.

**Approach A (primary):** Direct asset analysis via `VideoContext_AssetId` — no indexing needed.

**Approach B (fallback):** If direct analysis fails, create a Pegasus index, index assets, then analyze.

In [4]:
PEGASUS_PROMPT = (
    "Describe the main activity, setting, and key objects visible "
    "in this video in one concise sentence."
)
POLL_INTERVAL = 10.0
RATE_LIMIT_WAIT = 30

use_indexing = False
index_id = None
cluster_labels = {}

for cid in sorted(representatives.keys()):
    samples = representatives[cid]
    descriptions = []

    print(f"Cluster {cid}:")

    for sample in samples:
        filepath = sample.filepath
        filename = os.path.basename(filepath)
        print(f"  {filename}: ", end="", flush=True)
        start = time.time()

        # Upload asset
        try:
            with open(filepath, "rb") as f:
                asset = client.assets.create(method="direct", file=f)
        except Exception as e:
            print(f"UPLOAD FAILED: {e}")
            continue

        # Generate description
        success = False
        for attempt in range(2):
            try:
                if use_indexing:
                    # Approach B: index then analyze
                    if index_id is None:
                        timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
                        print(f"creating index ... ", end="", flush=True)
                        idx = client.indexes.create(
                            index_name=f"gap-analyzer-{timestamp}",
                            models=[IndexesCreateRequestModelsItem(
                                model_name="pegasus1.2",
                                model_options=["visual", "audio"],
                            )],
                        )
                        index_id = idx.id
                        print(f"({index_id}) ", end="", flush=True)

                    resp = client.indexes.indexed_assets.create(
                        index_id=index_id, asset_id=asset.id,
                    )
                    indexed_asset_id = resp.id
                    print(f"indexing ... ", end="", flush=True)
                    while True:
                        detail = client.indexes.indexed_assets.retrieve(
                            index_id, indexed_asset_id,
                        )
                        if detail.status == "ready":
                            break
                        if detail.status == "failed":
                            raise RuntimeError("Indexing failed")
                        time.sleep(POLL_INTERVAL)

                    result = client.analyze(
                        video_id=indexed_asset_id,
                        prompt=PEGASUS_PROMPT,
                        temperature=0.2,
                    )
                else:
                    # Approach A: direct asset analysis
                    print(f"analyzing ... ", end="", flush=True)
                    result = client.analyze(
                        video=VideoContext_AssetId(asset_id=asset.id),
                        prompt=PEGASUS_PROMPT,
                        temperature=0.2,
                    )

                text = result.data
                elapsed = time.time() - start
                if text:
                    descriptions.append(text.strip())
                    print(f"OK ({elapsed:.1f}s)")
                else:
                    print(f"empty ({elapsed:.1f}s)")
                success = True
                break

            except Exception as e:
                elapsed = time.time() - start
                err = str(e).lower()
                if "429" in str(e) or "rate" in err or "too many" in err:
                    if attempt == 0:
                        print(f"rate limited, waiting {RATE_LIMIT_WAIT}s ... ", end="", flush=True)
                        time.sleep(RATE_LIMIT_WAIT)
                        continue
                elif not use_indexing:
                    print(f"direct failed ({elapsed:.1f}s): {e}")
                    print("  Switching to index-based approach...")
                    use_indexing = True
                    continue
                print(f"FAILED ({elapsed:.1f}s): {e}")
                break

    # Combine into label
    if len(descriptions) >= 2:
        cluster_labels[cid] = f"{descriptions[0]}; {descriptions[1]}"
    elif len(descriptions) == 1:
        cluster_labels[cid] = descriptions[0]
    else:
        cluster_labels[cid] = f"Cluster {cid}"
        print("  WARNING: No descriptions, using fallback label")

    print(f"  -> Label: {cluster_labels[cid]}\n")

Cluster 0:
  0_tr103.mp4: analyzing ... OK (53.7s)
  0_tr100.mp4: analyzing ... OK (36.0s)
  -> Label: The video captures a static shot of a bustling factory floor, where industrial machinery and equipment are scattered across the space, and a worker is seen walking in the background, moving through the environment with purpose.; The video captures a static view of a bustling factory floor at 00:00, featuring industrial machinery, conveyor systems, and scattered equipment, with a partially visible worker in the background, all set under a high-ceilinged, dimly lit environment that emphasizes the operational intensity of the space.

Cluster 1:
  1_tr13.mp4: analyzing ... OK (26.3s)
  1_tr14.mp4: analyzing ... OK (14.8s)
  -> Label: The video captures a factory environment where a worker in a safety helmet operates industrial machinery, with another worker positioned near the entrance, surrounded by various machines and materials, highlighting a scene of routine manufacturing operations.

## 5. Apply Labels to All Samples

Write `cluster_label` to every sample based on its `cluster_id`.

In [5]:
updated = 0
for sample in dataset:
    try:
        cid = sample["cluster_id"]
    except (KeyError, AttributeError):
        continue
    if cid is not None and cid in cluster_labels:
        sample["cluster_label"] = cluster_labels[cid]
        sample.save()
        updated += 1

dataset.save()
print(f"{updated} samples updated with cluster_label")

30 samples updated with cluster_label


## 6. Summary

Display all cluster labels with sample counts.

In [6]:
print("=" * 60)
print("CLUSTER DESCRIPTION SUMMARY")
print("=" * 60)

for cid in sorted(cluster_labels.keys()):
    count = sum(
        1 for s in dataset
        if s.get_field("cluster_id") is not None and s["cluster_id"] == cid
    )
    print(f"\n  Cluster {cid} ({count} samples):")
    print(f"    \"{cluster_labels[cid]}\"")

# Sanity check
missing = 0
for sample in dataset:
    try:
        cid = sample["cluster_id"]
    except (KeyError, AttributeError):
        continue
    if cid is not None:
        try:
            label = sample["cluster_label"]
            if label is None:
                missing += 1
        except (KeyError, AttributeError):
            missing += 1

print(f"\n{'All' if missing == 0 else f'{missing} missing'} clustered samples have cluster_label")

CLUSTER DESCRIPTION SUMMARY

  Cluster 0 (14 samples):
    "The video captures a static shot of a bustling factory floor, where industrial machinery and equipment are scattered across the space, and a worker is seen walking in the background, moving through the environment with purpose.; The video captures a static view of a bustling factory floor at 00:00, featuring industrial machinery, conveyor systems, and scattered equipment, with a partially visible worker in the background, all set under a high-ceilinged, dimly lit environment that emphasizes the operational intensity of the space."

  Cluster 1 (16 samples):
    "The video captures a factory environment where a worker in a safety helmet operates industrial machinery, with another worker positioned near the entrance, surrounded by various machines and materials, highlighting a scene of routine manufacturing operations.; The video captures a factory environment where a worker walks through a corridor lined with industrial machine

## 7. Launch FiftyOne App

Browse the dataset with cluster labels visible in the App.

In [8]:
session = fo.launch_app(dataset)